# Solar Filament 2026 — Full 5-Fold PQ Pipeline

## tl;dr

This is a complete train → grouped OOF → test inference notebook:

1. group physical observations before splitting annotator records;
2. choose the medoid annotation record for instance training;
3. train five single-class YOLO11 segmentation detectors;
4. train five seed-conditioned crop refiners;
5. evaluate the complete detector → refiner → post-processing path with Panoptic Quality;
6. ensemble all five folds and write `submission.csv`.

The notebook uses only official train annotations and official H-alpha images. It does **not** use external test labels, MAGFiLO mirrors containing test annotations, or leaderboard exploits.

## August 7 metric update

The [competition evaluation page](https://www.kaggle.com/competitions/filament-segmentation-2026/overview/evaluation) changed the primary metric from Dice to **Panoptic Quality (PQ)** on 7 August 2026:

\[
PQ = \frac{\sum_{(y,\hat y)\in TP} IoU(y,\hat y)}{|TP| + 0.5|FP| + 0.5|FN|}.
\]

Following the original PQ paper cited by the organizers, a pair is a true positive only when `IoU > 0.5`. The notebook reports:

- **SQ**: mean IoU of matched instances;
- **RQ**: recognition quality, `TP / (TP + 0.5 FP + 0.5 FN)`;
- **PQ = SQ × RQ**.

This materially changes model selection: every missing and extra filament matters, and fragmented/merged predictions usually fail the `IoU > 0.5` match.

## Kaggle run contract

- Attach the official `filament-segmentation-2026` competition data.
- Enable a GPU; two T4s are supported for YOLO training.
- Enable Internet for first-run package installation and pretrained weights.
- Full mode trains five detector/refiner pairs. Checkpoints are reused when cells are rerun.
- Set `DEBUG=True` for a two-epoch smoke run on small subsets.

In [ ]:
from pathlib import Path
import importlib.util, os, shutil, subprocess, sys

KAGGLE_DATA_AVAILABLE = Path('/kaggle/input').exists()
if KAGGLE_DATA_AVAILABLE:
    wheel = next(Path('/kaggle/input').glob('**/ultralytics-8.4.103-py3-none-any.whl'))
    dependency_dir = wheel.parent
    if importlib.util.find_spec('ultralytics') is None:
        wheels = [str(path) for path in sorted(dependency_dir.glob('*.whl'))]
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', *wheels])
    torch_home = Path('/kaggle/working/torch_cache')
    checkpoint_dir = torch_home / 'hub' / 'checkpoints'
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(dependency_dir / 'resnet18-f37072fd.pth', checkpoint_dir / 'resnet18-f37072fd.pth')
    shutil.copy2(dependency_dir / 'yolo11s-seg.pt', Path('/kaggle/working/yolo11s-seg.pt'))
    os.environ['TORCH_HOME'] = str(torch_home)
print({'kaggle_data_available': KAGGLE_DATA_AVAILABLE, 'offline_dependencies': KAGGLE_DATA_AVAILABLE})

## Pipeline

### 1. Configuration, COCO indexing, medoid annotators, and grouped folds

In [ ]:
"""Full leakage-safe Solar Filament train/OOF/test pipeline for Kaggle.

Primary metric: Panoptic Quality (competition update, 7 Aug 2026).
The module uses only official train annotations and H-alpha images.
"""

from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import random
import shutil
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Mapping, Sequence

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pycocotools import mask as mask_utils
from scipy.optimize import linear_sum_assignment
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18


@dataclass
class CFG:
    seed: int = 20260729
    n_folds: int = 5
    image_size: int = 1024
    detector_model: str = "yolo11s-seg.pt"
    detector_epochs: int = 30
    detector_batch: int = 4
    detector_patience: int = 7
    detector_conf: float = 0.20
    detector_fallback_conf: float = 0.05
    detector_iou: float = 0.60
    mask_ratio: int = 2
    refiner_crop: int = 256
    refiner_context: float = 1.8
    refiner_min_crop: int = 96
    refiner_epochs: int = 8
    refiner_batch: int = 16
    refiner_lr: float = 3e-4
    refiner_threshold: float = 0.65
    minimum_area: int = 96
    pq_match_iou: float = 0.50
    ensemble_iou: float = 0.50
    minimum_fold_support: int = 2
    singleton_score: float = 0.90
    num_workers: int = 2
    use_tta: bool = True
    save_oof_masks: bool = False
    debug: bool = False


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def discover_competition_paths() -> dict[str, Path]:
    roots = sorted(Path("/kaggle/input").glob("**/MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    if len(roots) != 1:
        raise RuntimeError(f"Expected one official train JSON, found {roots}")
    annotation_path = roots[0]
    data_root = annotation_path.parent.parent
    train_dir = data_root / "train" / "train_images"
    test_dir = data_root / "test" / "test_images"
    if not train_dir.is_dir() or not test_dir.is_dir():
        raise FileNotFoundError(f"Invalid competition tree under {data_root}")
    return {"root": data_root, "json": annotation_path, "train": train_dir, "test": test_dir}


def load_coco(annotation_path: Path) -> dict[str, Any]:
    with annotation_path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def build_coco_indices(coco: Mapping[str, Any]) -> dict[str, Any]:
    images_by_id = {str(image["id"]): image for image in coco["images"]}
    annotations_by_image: dict[str, list[Mapping[str, Any]]] = {}
    for annotation in coco["annotations"]:
        annotations_by_image.setdefault(
            str(annotation["image_id"]), []
        ).append(annotation)
    image_ids_by_file: dict[str, list[str]] = {}
    for image_id, image in images_by_id.items():
        image_ids_by_file.setdefault(image["file_name"], []).append(image_id)
    for filename in image_ids_by_file:
        image_ids_by_file[filename].sort()
    return {
        "images_by_id": images_by_id,
        "annotations_by_image": annotations_by_image,
        "image_ids_by_file": image_ids_by_file,
    }


def annotation_to_rle(annotation: Mapping[str, Any], height: int, width: int) -> dict[str, Any]:
    segmentation = annotation.get("segmentation")
    if isinstance(segmentation, dict):
        rle = dict(segmentation)
        if isinstance(rle.get("counts"), list):
            rle = mask_utils.frPyObjects(rle, height, width)
        return rle
    polygons = segmentation or []
    if not polygons:
        return mask_utils.encode(np.asfortranarray(np.zeros((height, width), dtype=np.uint8)))
    rles = mask_utils.frPyObjects(polygons, height, width)
    return mask_utils.merge(rles)


def annotation_to_mask(annotation: Mapping[str, Any], height: int, width: int) -> np.ndarray:
    return mask_utils.decode(annotation_to_rle(annotation, height, width)).astype(bool)


def semantic_rle_for_record(
    image_id: int,
    images_by_id: Mapping[int, Mapping[str, Any]],
    annotations_by_image: Mapping[int, Sequence[Mapping[str, Any]]],
) -> dict[str, Any]:
    image = images_by_id[image_id]
    rles = [
        annotation_to_rle(annotation, int(image["height"]), int(image["width"]))
        for annotation in annotations_by_image.get(image_id, [])
        if not annotation.get("iscrowd", 0)
    ]
    if not rles:
        return mask_utils.encode(
            np.asfortranarray(np.zeros((int(image["height"]), int(image["width"])), dtype=np.uint8))
        )
    return mask_utils.merge(rles)


def choose_medoid_annotators(indices: Mapping[str, Any]) -> dict[str, int]:
    """Choose the annotator closest to the others, avoiding max-area bias."""
    selected: dict[str, int] = {}
    for filename, image_ids in indices["image_ids_by_file"].items():
        if len(image_ids) == 1:
            selected[filename] = image_ids[0]
            continue
        union_rles = [
            semantic_rle_for_record(
                image_id, indices["images_by_id"], indices["annotations_by_image"]
            )
            for image_id in image_ids
        ]
        pairwise_iou = np.asarray(mask_utils.iou(union_rles, union_rles, [0] * len(union_rles)))
        mean_agreement = (pairwise_iou.sum(axis=1) - 1.0) / max(len(image_ids) - 1, 1)
        selected[filename] = image_ids[int(np.argmax(mean_agreement))]
    return selected


def build_observation_table(indices: Mapping[str, Any]) -> pd.DataFrame:
    rows = []
    for filename, image_ids in sorted(indices["image_ids_by_file"].items()):
        counts = [len(indices["annotations_by_image"].get(image_id, [])) for image_id in image_ids]
        rows.append(
            {
                "filename": filename,
                "site": Path(filename).stem[-2:],
                "n_annotators": len(image_ids),
                "mean_instances": float(np.mean(counts)),
                "max_instances": max(counts, default=0),
            }
        )
    return pd.DataFrame(rows)


def assign_grouped_folds(observations: pd.DataFrame, cfg: CFG) -> pd.DataFrame:
    frame = observations.copy().sort_values("filename").reset_index(drop=True)
    count_bin = pd.qcut(
        frame["mean_instances"].rank(method="first"),
        q=min(5, len(frame)), labels=False, duplicates="drop",
    ).astype(int)
    frame["stratum"] = frame["site"] + "_q" + count_bin.astype(str)
    frame["fold"] = -1
    fold_sizes = np.zeros(cfg.n_folds, dtype=int)
    rng = np.random.default_rng(cfg.seed)
    strata = sorted(frame.groupby("stratum").groups.items(), key=lambda item: (-len(item[1]), item[0]))
    for _, row_indices in strata:
        row_indices = np.asarray(list(row_indices), dtype=int)
        rng.shuffle(row_indices)
        for row_index in row_indices:
            choices = np.flatnonzero(fold_sizes == fold_sizes.min())
            fold = int(rng.choice(choices))
            frame.loc[row_index, "fold"] = fold
            fold_sizes[fold] += 1
    assert frame["filename"].is_unique
    assert (frame["fold"] >= 0).all()
    return frame

### 2. Single-class YOLO dataset and detector training

In [ ]:
def mask_to_yolo_polygon(mask: np.ndarray, width: int, height: int) -> str | None:
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    contour = max(contours, key=cv2.contourArea).reshape(-1, 2)
    if len(contour) < 3:
        return None
    contour = contour.astype(np.float32)
    contour[:, 0] = np.clip(contour[:, 0] / width, 0, 1)
    contour[:, 1] = np.clip(contour[:, 1] / height, 0, 1)
    return "0 " + " ".join(f"{value:.6f}" for value in contour.ravel())


def write_yolo_split(
    split_dir: Path,
    filenames: Sequence[str],
    train_image_dir: Path,
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
) -> None:
    image_output = split_dir / "images"
    label_output = split_dir / "labels"
    image_output.mkdir(parents=True, exist_ok=True)
    label_output.mkdir(parents=True, exist_ok=True)
    for filename in filenames:
        source = train_image_dir / filename
        destination = image_output / filename
        if not destination.exists():
            os.symlink(source, destination)
        image_id = selected_image_ids[filename]
        info = indices["images_by_id"][image_id]
        lines = []
        for annotation in indices["annotations_by_image"].get(image_id, []):
            if annotation.get("iscrowd", 0):
                continue
            mask = annotation_to_mask(annotation, int(info["height"]), int(info["width"]))
            line = mask_to_yolo_polygon(mask, int(info["width"]), int(info["height"]))
            if line is not None:
                lines.append(line)
        (label_output / f"{Path(filename).stem}.txt").write_text("\n".join(lines), encoding="utf-8")


def prepare_yolo_fold(
    fold: int,
    fold_manifest: pd.DataFrame,
    train_image_dir: Path,
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    work_dir: Path,
) -> Path:
    fold_dir = work_dir / "yolo_data" / f"fold_{fold}"
    marker = fold_dir / ".ready"
    if marker.exists():
        return fold_dir / "data.yaml"
    train_files = fold_manifest.loc[fold_manifest["fold"] != fold, "filename"].tolist()
    val_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    write_yolo_split(fold_dir / "train", train_files, train_image_dir, selected_image_ids, indices)
    write_yolo_split(fold_dir / "val", val_files, train_image_dir, selected_image_ids, indices)
    yaml_text = (
        f"path: {fold_dir}\n"
        "train: train/images\n"
        "val: val/images\n"
        "names:\n  0: filament\n"
    )
    yaml_path = fold_dir / "data.yaml"
    yaml_path.write_text(yaml_text, encoding="utf-8")
    marker.write_text("ready", encoding="utf-8")
    return yaml_path


def train_detector(fold: int, data_yaml: Path, cfg: CFG, work_dir: Path) -> Path:
    from ultralytics import YOLO

    output_dir = work_dir / "models" / f"fold_{fold}"
    output_dir.mkdir(parents=True, exist_ok=True)
    final_path = output_dir / "detector.pt"
    if final_path.exists():
        return final_path
    epochs = 2 if cfg.debug else cfg.detector_epochs
    model = YOLO(cfg.detector_model)
    gpu_count = torch.cuda.device_count()
    device: int | str = ",".join(map(str, range(gpu_count))) if gpu_count > 1 else (0 if gpu_count == 1 else "cpu")
    model.train(
        data=str(data_yaml), epochs=epochs, imgsz=cfg.image_size,
        batch=cfg.detector_batch, device=device, patience=cfg.detector_patience,
        seed=cfg.seed + fold, single_cls=True, mask_ratio=cfg.mask_ratio,
        overlap_mask=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.25,
        fliplr=0.5, flipud=0.5, degrees=12, scale=0.30, translate=0.08,
        mosaic=0.25, close_mosaic=5, project=str(work_dir / "detector_runs"),
        name=f"fold_{fold}", exist_ok=True, verbose=True,
    )
    shutil.copy2(str(model.trainer.best), final_path)
    return final_path

### 3. Seed-conditioned refiner training

In [ ]:
def square_bounds(mask: np.ndarray, context: float, minimum: int) -> tuple[int, int, int, int]:
    height, width = mask.shape
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return 0, 0, width, height
    center_x = (xs.min() + xs.max() + 1) / 2
    center_y = (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(np.ptp(xs) + 1, np.ptp(ys) + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(center_x - side / 2)), int(round(center_y - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return x0, y0, x1, y1


def jitter_seed_mask(mask: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    seed = mask.astype(np.uint8)
    radius = int(rng.integers(-5, 6))
    if radius != 0:
        kernel_size = 2 * abs(radius) + 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        seed = cv2.dilate(seed, kernel) if radius > 0 else cv2.erode(seed, kernel)
    shift_x, shift_y = int(rng.integers(-8, 9)), int(rng.integers(-8, 9))
    matrix = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
    seed = cv2.warpAffine(seed, matrix, (seed.shape[1], seed.shape[0]), flags=cv2.INTER_NEAREST)
    return seed.astype(bool) if seed.any() else mask.astype(bool)


class RefinerDataset(Dataset):
    def __init__(
        self,
        filenames: Sequence[str],
        image_dir: Path,
        selected_image_ids: Mapping[str, int],
        indices: Mapping[str, Any],
        cfg: CFG,
        augment: bool,
    ) -> None:
        self.image_dir, self.indices, self.cfg, self.augment = image_dir, indices, cfg, augment
        self.samples = []
        for filename in filenames:
            image_id = selected_image_ids[filename]
            for annotation in indices["annotations_by_image"].get(image_id, []):
                if not annotation.get("iscrowd", 0):
                    self.samples.append((filename, image_id, annotation))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        filename, image_id, annotation = self.samples[index]
        info = self.indices["images_by_id"][image_id]
        image = cv2.imread(str(self.image_dir / filename), cv2.IMREAD_GRAYSCALE)
        target = annotation_to_mask(annotation, int(info["height"]), int(info["width"]))
        rng = np.random.default_rng(self.cfg.seed + index + random.randint(0, 1_000_000))
        seed = jitter_seed_mask(target, rng) if self.augment else target
        x0, y0, x1, y1 = square_bounds(seed, self.cfg.refiner_context, self.cfg.refiner_min_crop)
        image_crop = cv2.resize(image[y0:y1, x0:x1], (self.cfg.refiner_crop,) * 2, interpolation=cv2.INTER_LINEAR)
        target_crop = cv2.resize(target[y0:y1, x0:x1].astype(np.uint8), (self.cfg.refiner_crop,) * 2, interpolation=cv2.INTER_NEAREST)
        seed_crop = cv2.resize(seed[y0:y1, x0:x1].astype(np.uint8), (self.cfg.refiner_crop,) * 2, interpolation=cv2.INTER_NEAREST)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image_crop)
        features = np.stack([image_crop, clahe, seed_crop * 255], axis=0).astype(np.float32) / 255.0
        if self.augment:
            k = random.randrange(4)
            features = np.rot90(features, k, axes=(1, 2)).copy()
            target_crop = np.rot90(target_crop, k).copy()
            if random.random() < 0.5:
                features = features[:, :, ::-1].copy(); target_crop = target_crop[:, ::-1].copy()
            gain = random.uniform(0.9, 1.1)
            features[:2] = np.clip(features[:2] * gain, 0, 1)
        return torch.from_numpy(features), torch.from_numpy(target_crop[None].astype(np.float32))


class ConvBlock(nn.Sequential):
    def __init__(self, input_channels: int, output_channels: int) -> None:
        super().__init__(
            nn.Conv2d(input_channels, output_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(output_channels), nn.ReLU(inplace=True),
            nn.Conv2d(output_channels, output_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(output_channels), nn.ReLU(inplace=True),
        )


class SeedRefiner(nn.Module):
    def __init__(self, pretrained: bool = True) -> None:
        super().__init__()
        encoder = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.pool, self.layer1, self.layer2, self.layer3, self.layer4 = (
            encoder.maxpool, encoder.layer1, encoder.layer2, encoder.layer3, encoder.layer4
        )
        self.up4, self.dec4 = nn.ConvTranspose2d(512, 256, 2, 2), ConvBlock(512, 256)
        self.up3, self.dec3 = nn.ConvTranspose2d(256, 128, 2, 2), ConvBlock(256, 128)
        self.up2, self.dec2 = nn.ConvTranspose2d(128, 64, 2, 2), ConvBlock(128, 64)
        self.up1, self.dec1 = nn.ConvTranspose2d(64, 32, 2, 2), ConvBlock(96, 32)
        self.up0, self.head = nn.ConvTranspose2d(32, 16, 2, 2), nn.Conv2d(16, 1, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, tensor: torch.Tensor) -> torch.Tensor:
        tensor = (tensor - self.mean) / self.std
        x0 = self.stem(tensor); x1 = self.layer1(self.pool(x0)); x2 = self.layer2(x1)
        x3 = self.layer3(x2); x4 = self.layer4(x3)
        x = self.dec4(torch.cat([self.up4(x4), x3], dim=1))
        x = self.dec3(torch.cat([self.up3(x), x2], dim=1))
        x = self.dec2(torch.cat([self.up2(x), x1], dim=1))
        x = self.dec1(torch.cat([self.up1(x), x0], dim=1))
        return self.head(self.up0(x))


def soft_dice_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    probabilities = logits.sigmoid()
    intersection = (probabilities * targets).sum(dim=(1, 2, 3))
    denominator = probabilities.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return (1 - (2 * intersection + 1) / (denominator + 1)).mean()


def train_refiner(
    fold: int,
    fold_manifest: pd.DataFrame,
    image_dir: Path,
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
) -> Path:
    output_dir = work_dir / "models" / f"fold_{fold}"
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint = output_dir / "refiner.pt"
    if checkpoint.exists():
        return checkpoint
    train_files = fold_manifest.loc[fold_manifest["fold"] != fold, "filename"].tolist()
    val_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
    if cfg.debug:
        train_files, val_files = train_files[:24], val_files[:8]
    train_dataset = RefinerDataset(train_files, image_dir, selected_image_ids, indices, cfg, True)
    val_dataset = RefinerDataset(val_files, image_dir, selected_image_ids, indices, cfg, False)
    train_loader = DataLoader(train_dataset, batch_size=cfg.refiner_batch, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=cfg.refiner_batch * 2, shuffle=False,
                            num_workers=cfg.num_workers, pin_memory=True)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = SeedRefiner(pretrained=True).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.refiner_lr, weight_decay=1e-4)
    epochs = 2 if cfg.debug else cfg.refiner_epochs
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    best_dice = -1.0
    for epoch in range(epochs):
        model.train()
        for features, targets in train_loader:
            features, targets = features.to(device), targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(features)
                loss = 0.45 * F.binary_cross_entropy_with_logits(logits, targets) + 0.55 * soft_dice_loss(logits, targets)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        model.eval(); dice_total = sample_count = 0
        with torch.no_grad():
            for features, targets in val_loader:
                features, targets = features.to(device), targets.to(device).bool()
                predictions = model(features).sigmoid() >= cfg.refiner_threshold
                intersection = (predictions & targets).sum(dim=(1, 2, 3)).float()
                denominator = predictions.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
                dice_total += ((2 * intersection + 1e-7) / (denominator + 1e-7)).sum().item()
                sample_count += len(features)
        validation_dice = dice_total / max(sample_count, 1)
        print(f"fold={fold} refiner_epoch={epoch + 1} val_crop_dice={validation_dice:.5f}")
        if validation_dice > best_dice:
            best_dice = validation_dice
            torch.save(model.state_dict(), checkpoint)
        scheduler.step()
    return checkpoint

### 4. Topology-safe end-to-end inference

In [ ]:
def component_count(mask: np.ndarray) -> int:
    return max(cv2.connectedComponents(mask.astype(np.uint8), connectivity=8)[0] - 1, 0)


def binary_dice(first: np.ndarray, second: np.ndarray) -> float:
    intersection = np.logical_and(first, second).sum()
    return float((2 * intersection + 1e-7) / (first.sum() + second.sum() + 1e-7))


@torch.no_grad()
def refine_seed(
    image: np.ndarray,
    seed_mask: np.ndarray,
    model: SeedRefiner,
    device: torch.device,
    cfg: CFG,
) -> tuple[np.ndarray, float]:
    x0, y0, x1, y1 = square_bounds(seed_mask, cfg.refiner_context, cfg.refiner_min_crop)
    image_crop = cv2.resize(image[y0:y1, x0:x1], (cfg.refiner_crop,) * 2, interpolation=cv2.INTER_LINEAR)
    seed_crop = cv2.resize(seed_mask[y0:y1, x0:x1].astype(np.uint8), (cfg.refiner_crop,) * 2, interpolation=cv2.INTER_NEAREST)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image_crop)
    features = np.stack([image_crop, clahe, seed_crop * 255], axis=0).astype(np.float32) / 255.0
    tensor = torch.from_numpy(features[None]).to(device)
    transforms = [lambda value: value]
    inverses = [lambda value: value]
    if cfg.use_tta:
        transforms += [lambda value: value.flip(-1), lambda value: value.flip(-2), lambda value: value.flip((-2, -1))]
        inverses += [lambda value: value.flip(-1), lambda value: value.flip(-2), lambda value: value.flip((-2, -1))]
    probabilities = []
    for transform, inverse in zip(transforms, inverses):
        probability = inverse(model(transform(tensor)).sigmoid())[0, 0].cpu().numpy()
        probabilities.append(probability)
    base_probability = probabilities[0]
    candidate_probability = np.mean(probabilities, axis=0)
    base_mask = base_probability >= cfg.refiner_threshold
    candidate_mask = candidate_probability >= cfg.refiner_threshold
    use_candidate = True
    if base_mask.any():
        area_ratio = candidate_mask.sum() / max(base_mask.sum(), 1)
        use_candidate = (
            0.60 <= area_ratio <= 1.70
            and binary_dice(base_mask, candidate_mask) >= 0.62
            and component_count(candidate_mask) <= component_count(base_mask) + 1
        )
    probability = candidate_probability if use_candidate else base_probability
    resized_probability = cv2.resize(probability, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
    crop_mask = resized_probability >= cfg.refiner_threshold
    full_mask = np.zeros_like(seed_mask, dtype=bool)
    full_mask[y0:y1, x0:x1] = crop_mask
    if not full_mask.any():
        full_mask = seed_mask.astype(bool)
    confidence = float(resized_probability[crop_mask].mean()) if crop_mask.any() else 0.0
    return full_mask, confidence


def mask_iou(first: np.ndarray, second: np.ndarray) -> float:
    intersection = np.logical_and(first, second).sum()
    union = np.logical_or(first, second).sum()
    return float(intersection / union) if union else 0.0


def remove_duplicate_seeds(masks: list[np.ndarray], scores: list[float], threshold: float = 0.60) -> tuple[list[np.ndarray], list[float]]:
    kept_masks, kept_scores = [], []
    for index in np.argsort(scores)[::-1]:
        mask = masks[int(index)]
        if all(mask_iou(mask, kept) < threshold for kept in kept_masks):
            kept_masks.append(mask); kept_scores.append(float(scores[int(index)]))
    return kept_masks, kept_scores


def resolve_overlaps(masks: list[np.ndarray], scores: list[float], minimum_area: int) -> tuple[list[np.ndarray], list[float]]:
    if not masks:
        return [], []
    assigned = np.zeros_like(masks[0], dtype=bool)
    output_masks, output_scores = [], []
    for index in np.argsort(scores)[::-1]:
        clean = masks[int(index)].astype(bool) & ~assigned
        if clean.sum() >= minimum_area:
            output_masks.append(clean); output_scores.append(float(scores[int(index)])); assigned |= clean
    return output_masks, output_scores


def load_fold_models(fold: int, cfg: CFG, work_dir: Path) -> tuple[Any, SeedRefiner, torch.device]:
    from ultralytics import YOLO

    model_dir = work_dir / "models" / f"fold_{fold}"
    detector = YOLO(str(model_dir / "detector.pt"))
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    refiner = SeedRefiner(pretrained=False)
    refiner.load_state_dict(torch.load(model_dir / "refiner.pt", map_location=device))
    refiner.to(device).eval()
    return detector, refiner, device


def classical_fallback_seed(image: np.ndarray, minimum_area: int) -> np.ndarray:
    """Image-only last resort so every test observation has a valid prediction."""
    blurred = cv2.GaussianBlur(image, (0, 0), 25)
    darkness = blurred.astype(np.float32) - image.astype(np.float32)
    disk = image > 12
    values = darkness[disk]
    if values.size == 0:
        disk = np.ones_like(image, dtype=bool)
        values = darkness.ravel()
    threshold = float(np.quantile(values, 0.995))
    candidate = ((darkness >= threshold) & disk).astype(np.uint8)
    candidate = cv2.morphologyEx(
        candidate, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    )
    count, labels, stats, _ = cv2.connectedComponentsWithStats(candidate, 8)
    valid = [index for index in range(1, count) if stats[index, cv2.CC_STAT_AREA] >= minimum_area]
    if not valid:
        valid = [1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))] if count > 1 else []
    if not valid:
        fallback = np.zeros_like(image, dtype=np.uint8)
        center_y, center_x = np.unravel_index(int(np.argmax(darkness)), darkness.shape)
        cv2.circle(fallback, (int(center_x), int(center_y)), 6, 1, -1)
        return fallback.astype(bool)
    best = max(valid, key=lambda index: stats[index, cv2.CC_STAT_AREA])
    return labels == best


@torch.no_grad()
def predict_instances(
    image_path: Path,
    detector: Any,
    refiner: SeedRefiner,
    device: torch.device,
    cfg: CFG,
) -> tuple[list[np.ndarray], list[float]]:
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    result = detector.predict(
        str(image_path), imgsz=cfg.image_size, conf=cfg.detector_conf,
        iou=cfg.detector_iou, agnostic_nms=True, retina_masks=True, verbose=False,
    )[0]
    if result.masks is None:
        result = detector.predict(
            str(image_path), imgsz=cfg.image_size, conf=cfg.detector_fallback_conf,
            iou=cfg.detector_iou, agnostic_nms=True, retina_masks=True, verbose=False,
        )[0]
    if result.masks is None:
        fallback_seed = classical_fallback_seed(image, cfg.minimum_area)
        fallback_mask, fallback_confidence = refine_seed(image, fallback_seed, refiner, device, cfg)
        return [fallback_mask], [0.01 * fallback_confidence]
    seed_masks = [mask.cpu().numpy().astype(bool) for mask in result.masks.data]
    if seed_masks and seed_masks[0].shape != image.shape:
        seed_masks = [cv2.resize(mask.astype(np.uint8), image.shape[::-1], interpolation=cv2.INTER_NEAREST).astype(bool) for mask in seed_masks]
    detector_scores = result.boxes.conf.detach().cpu().numpy().astype(float).tolist()
    seed_masks, detector_scores = remove_duplicate_seeds(seed_masks, detector_scores)
    masks, scores = [], []
    for seed_mask, detector_score in zip(seed_masks, detector_scores):
        if seed_mask.sum() < cfg.minimum_area:
            continue
        refined_mask, refiner_score = refine_seed(image, seed_mask, refiner, device, cfg)
        masks.append(refined_mask); scores.append(float(detector_score * refiner_score))
    return resolve_overlaps(masks, scores, cfg.minimum_area)

### 5. August 7 Panoptic Quality and grouped OOF

In [ ]:
def pairwise_iou(gt_masks: Sequence[np.ndarray], pred_masks: Sequence[np.ndarray]) -> np.ndarray:
    if not gt_masks or not pred_masks:
        return np.zeros((len(gt_masks), len(pred_masks)), dtype=np.float64)
    gt_rles = [mask_utils.encode(np.asfortranarray(mask.astype(np.uint8))) for mask in gt_masks]
    pred_rles = [mask_utils.encode(np.asfortranarray(mask.astype(np.uint8))) for mask in pred_masks]
    return np.asarray(mask_utils.iou(gt_rles, pred_rles, [0] * len(pred_rles)), dtype=np.float64)


def panoptic_quality(
    gt_masks: Sequence[np.ndarray],
    pred_masks: Sequence[np.ndarray],
    match_iou: float = 0.50,
) -> dict[str, float | int]:
    """August 7 metric. TP uses the original PQ criterion IoU > 0.5."""
    n_gt, n_pred = len(gt_masks), len(pred_masks)
    if n_gt == 0 and n_pred == 0:
        return {"pq": 1.0, "sq": 1.0, "rq": 1.0, "tp": 0, "fp": 0, "fn": 0, "sum_iou": 0.0}
    iou = pairwise_iou(gt_masks, pred_masks)
    if n_gt and n_pred:
        gt_indices, pred_indices = linear_sum_assignment(-iou)
        matched_ious = iou[gt_indices, pred_indices]
        matched_ious = matched_ious[matched_ious > match_iou]
    else:
        matched_ious = np.zeros(0, dtype=float)
    tp = int(len(matched_ious)); fp = int(n_pred - tp); fn = int(n_gt - tp)
    sum_iou = float(matched_ious.sum())
    denominator = tp + 0.5 * fp + 0.5 * fn
    pq = sum_iou / denominator if denominator else 1.0
    sq = sum_iou / tp if tp else 0.0
    rq = tp / denominator if denominator else 1.0
    return {"pq": pq, "sq": sq, "rq": rq, "tp": tp, "fp": fp, "fn": fn, "sum_iou": sum_iou}


def ground_truth_sets_for_file(filename: str, indices: Mapping[str, Any]) -> list[list[np.ndarray]]:
    result = []
    for image_id in indices["image_ids_by_file"][filename]:
        info = indices["images_by_id"][image_id]
        masks = [
            annotation_to_mask(annotation, int(info["height"]), int(info["width"]))
            for annotation in indices["annotations_by_image"].get(image_id, [])
            if not annotation.get("iscrowd", 0)
        ]
        result.append(masks)
    return result


def evaluate_prediction_for_file(
    filename: str,
    pred_masks: Sequence[np.ndarray],
    indices: Mapping[str, Any],
    cfg: CFG,
) -> list[dict[str, Any]]:
    rows = []
    for annotator_index, gt_masks in enumerate(ground_truth_sets_for_file(filename, indices)):
        rows.append({"filename": filename, "annotator": annotator_index,
                     **panoptic_quality(gt_masks, pred_masks, cfg.pq_match_iou),
                     "n_gt": len(gt_masks), "n_pred": len(pred_masks)})
    return rows

### 6. Fold orchestration

In [ ]:
def train_all_folds(
    fold_manifest: pd.DataFrame,
    paths: Mapping[str, Path],
    selected_image_ids: Mapping[str, int],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
) -> None:
    for fold in range(cfg.n_folds):
        print(f"\n===== TRAIN FOLD {fold}/{cfg.n_folds - 1} =====")
        data_yaml = prepare_yolo_fold(fold, fold_manifest, paths["train"], selected_image_ids, indices, work_dir)
        train_detector(fold, data_yaml, cfg, work_dir)
        train_refiner(fold, fold_manifest, paths["train"], selected_image_ids, indices, cfg, work_dir)
        gc.collect(); torch.cuda.empty_cache()


def run_oof(
    fold_manifest: pd.DataFrame,
    paths: Mapping[str, Path],
    indices: Mapping[str, Any],
    cfg: CFG,
    work_dir: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    annotator_rows = []
    oof_dir = work_dir / "oof_masks"; oof_dir.mkdir(parents=True, exist_ok=True)
    for fold in range(cfg.n_folds):
        detector, refiner, device = load_fold_models(fold, cfg, work_dir)
        validation_files = fold_manifest.loc[fold_manifest["fold"] == fold, "filename"].tolist()
        if cfg.debug: validation_files = validation_files[:8]
        for number, filename in enumerate(validation_files, 1):
            masks, scores = predict_instances(paths["train"] / filename, detector, refiner, device, cfg)
            rows = evaluate_prediction_for_file(filename, masks, indices, cfg)
            for row in rows: row["fold"] = fold
            annotator_rows.extend(rows)
            if cfg.save_oof_masks:
                shape = (2048, 2048)
                stack = np.stack(masks).astype(np.uint8) if masks else np.zeros((0, *shape), np.uint8)
                np.savez_compressed(oof_dir / f"{Path(filename).stem}.npz", masks=stack, scores=np.asarray(scores))
            if number % 20 == 0: print(f"fold={fold} OOF {number}/{len(validation_files)}")
        del detector, refiner; gc.collect(); torch.cuda.empty_cache()
    annotator_df = pd.DataFrame(annotator_rows)
    metrics = ["pq", "sq", "rq", "tp", "fp", "fn", "sum_iou", "n_gt", "n_pred"]
    observation_df = annotator_df.groupby(["filename", "fold"], as_index=False)[metrics].mean()
    annotator_df.to_csv(work_dir / "oof_by_annotator.csv", index=False)
    observation_df.to_csv(work_dir / "oof_by_observation.csv", index=False)
    return observation_df, annotator_df


def summarize_oof(observation_df: pd.DataFrame) -> pd.DataFrame:
    metrics = ["pq", "sq", "rq", "tp", "fp", "fn", "n_gt", "n_pred"]
    fold_summary = observation_df.groupby("fold")[metrics].mean().reset_index()
    overall = pd.DataFrame([{"fold": "overall", **observation_df[metrics].mean().to_dict()}])
    return pd.concat([fold_summary, overall], ignore_index=True)

### 7. Five-fold test ensemble and submission

In [ ]:
def fuse_fold_predictions(
    predictions: list[tuple[int, np.ndarray, float]],
    cfg: CFG,
) -> tuple[list[np.ndarray], list[float]]:
    clusters: list[list[tuple[int, np.ndarray, float]]] = []
    for item in sorted(predictions, key=lambda value: value[2], reverse=True):
        best_cluster, best_iou = None, 0.0
        for cluster in clusters:
            representative = cluster[0][1]
            overlap = mask_iou(item[1], representative)
            if overlap > best_iou:
                best_cluster, best_iou = cluster, overlap
        if best_cluster is not None and best_iou >= cfg.ensemble_iou:
            best_cluster.append(item)
        else:
            clusters.append([item])
    fused_masks, fused_scores = [], []
    for cluster in clusters:
        support = len({item[0] for item in cluster})
        maximum_score = max(item[2] for item in cluster)
        if support < cfg.minimum_fold_support and maximum_score < cfg.singleton_score:
            continue
        votes = np.stack([item[1] for item in cluster]).sum(axis=0)
        required_votes = max(1, math.ceil(len(cluster) / 2))
        fused = votes >= required_votes
        if fused.sum() >= cfg.minimum_area:
            fused_masks.append(fused); fused_scores.append(float(np.mean([item[2] for item in cluster])))
    return resolve_overlaps(fused_masks, fused_scores, cfg.minimum_area)


def encode_submission_mask(mask: np.ndarray) -> str:
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    return rle["counts"].decode("ascii")


def run_test_ensemble(paths: Mapping[str, Path], cfg: CFG, work_dir: Path) -> pd.DataFrame:
    test_files = sorted(path for path in paths["test"].iterdir() if path.suffix.lower() == ".jpeg")
    models = [load_fold_models(fold, cfg, work_dir) for fold in range(cfg.n_folds)]
    submission_rows = []
    for image_number, image_path in enumerate(test_files, 1):
        all_predictions = []
        for fold, (detector, refiner, device) in enumerate(models):
            masks, scores = predict_instances(image_path, detector, refiner, device, cfg)
            all_predictions.extend((fold, mask, score) for mask, score in zip(masks, scores))
        masks, _ = fuse_fold_predictions(all_predictions, cfg)
        image_id = image_path.stem
        for instance_index, mask in enumerate(masks, 1):
            submission_rows.append({
                "filament_id": f"{image_id}_{instance_index}",
                "segmentation_rle": encode_submission_mask(mask),
            })
        if image_number % 20 == 0:
            print(f"test {image_number}/{len(test_files)} rows={len(submission_rows)}")
    submission = pd.DataFrame(submission_rows, columns=["filament_id", "segmentation_rle"])
    if submission.empty or not submission["filament_id"].is_unique:
        raise RuntimeError("Invalid empty or duplicate-ID submission")
    submission.to_csv(work_dir / "submission.csv", index=False)
    return submission


def config_fingerprint(cfg: CFG, fold_manifest: pd.DataFrame) -> str:
    payload = json.dumps(asdict(cfg), sort_keys=True) + "\n" + "\n".join(
        f"{row.filename},{row.fold}" for row in fold_manifest.sort_values("filename").itertuples()
    )
    return hashlib.sha256(payload.encode()).hexdigest()


__all__ = [name for name in globals() if not name.startswith("_")]

## Configuration

The defaults are the intended full experiment, not a toy baseline.

In [ ]:
cfg = CFG(
    n_folds=5,
    image_size=1024,
    detector_model='yolo11s-seg.pt',
    detector_epochs=30,
    detector_batch=4,
    refiner_epochs=8,
    refiner_batch=16,
    detector_conf=0.20,
    detector_fallback_conf=0.05,
    refiner_threshold=0.65,
    minimum_area=96,
    pq_match_iou=0.50,
    minimum_fold_support=2,
    use_tta=True,
    save_oof_masks=False,
    debug=False,  # change to True for a quick Kaggle smoke run
)

RUN_TRAIN = True
RUN_OOF = True
RUN_TEST = True
WORK_DIR = Path('/kaggle/working/solar_filament_pq') if KAGGLE_DATA_AVAILABLE else Path('./solar_filament_pq_smoke')
WORK_DIR.mkdir(parents=True, exist_ok=True)
seed_everything(cfg.seed)
print(asdict(cfg))

## Checks

### 8. Metric edge cases required by PQ

In [ ]:
def _rectangle(y0, y1, x0, x1, size=16):
    mask = np.zeros((size, size), dtype=bool)
    mask[y0:y1, x0:x1] = True
    return mask

first = _rectangle(1, 5, 1, 5)
second = _rectangle(8, 14, 8, 14)
perfect = panoptic_quality([first, second], [second, first])
missing = panoptic_quality([first, second], [first])
extra = panoptic_quality([first], [first, second])

half_gt = _rectangle(2, 4, 2, 5)
half_pred = _rectangle(2, 4, 3, 6)  # exactly IoU=0.5; not a TP in original PQ
threshold_edge = panoptic_quality([half_gt], [half_pred])

fragment_gt = _rectangle(2, 6, 2, 10)
fragments = [_rectangle(2, 6, 2, 6), _rectangle(2, 6, 6, 10)]
fragmented = panoptic_quality([fragment_gt], fragments)

assert perfect['pq'] == 1.0
assert np.isclose(missing['pq'], 2 / 3) and missing['fn'] == 1
assert np.isclose(extra['pq'], 2 / 3) and extra['fp'] == 1
assert threshold_edge['tp'] == 0 and threshold_edge['pq'] == 0.0
assert fragmented['tp'] == 0 and fragmented['pq'] == 0.0

pd.DataFrame([perfect, missing, extra, threshold_edge, fragmented],
             index=['perfect', 'missing', 'extra', 'IoU=0.5', 'fragmented'])

## Data

### 9. Load official data, freeze folds, and audit leakage

In [ ]:
if not KAGGLE_DATA_AVAILABLE:
    paths = coco = indices = selected_image_ids = observations = fold_manifest = None
    print('Official competition input is not attached; training cells will skip.')
else:
    paths = discover_competition_paths()
    coco = load_coco(paths['json'])
    indices = build_coco_indices(coco)
    selected_image_ids = choose_medoid_annotators(indices)
    observations = build_observation_table(indices)
    fold_manifest = assign_grouped_folds(observations, cfg)
    fold_manifest.to_csv(WORK_DIR / 'fold_manifest.csv', index=False)
    fingerprint = config_fingerprint(cfg, fold_manifest)
    (WORK_DIR / 'run_config.json').write_text(
        json.dumps({'config': asdict(cfg), 'fingerprint': fingerprint}, indent=2), encoding='utf-8'
    )
    assert observations['filename'].nunique() == len(observations)
    assert fold_manifest.groupby('filename')['fold'].nunique().max() == 1
    print({
        'physical_observations': len(observations),
        'annotator_records': len(coco['images']),
        'train_annotations': len(coco['annotations']),
        'test_images': len(list(paths['test'].glob('*.jpeg'))),
        'fold_sizes': fold_manifest.groupby('fold').size().to_dict(),
        'fingerprint': fingerprint,
    })
    display(fold_manifest.groupby(['fold', 'site']).size().unstack(fill_value=0))

### Why medoid annotators?

Several physical observations have multiple expert records. Selecting the record with maximum instance count or area biases training toward over-segmentation. For each duplicated observation, this notebook selects the record with highest mean semantic IoU to the other annotators. All records remain available for OOF evaluation; only the medoid supplies detector/refiner training targets.

## Training

### 10. Train or resume five detector/refiner pairs

In [ ]:
if KAGGLE_DATA_AVAILABLE and RUN_TRAIN:
    train_all_folds(fold_manifest, paths, selected_image_ids, indices, cfg, WORK_DIR)
    print('All fold checkpoints are ready:', sorted((WORK_DIR / 'models').glob('fold_*/*.pt')))
else:
    print('Training skipped.')

## OOF Results

### 11. End-to-end grouped OOF

Validation uses predictions from the complete held-out detector → seed-conditioned refiner → overlap-resolution pipeline. It never evaluates refiner masks from ground-truth boxes. Every annotator is scored separately, then annotators are averaged inside the physical observation so duplicated records do not receive extra weight.

In [ ]:
if KAGGLE_DATA_AVAILABLE and RUN_OOF:
    observation_oof, annotator_oof = run_oof(fold_manifest, paths, indices, cfg, WORK_DIR)
    oof_summary = summarize_oof(observation_oof)
    oof_summary.to_csv(WORK_DIR / 'oof_summary.csv', index=False)
    display(oof_summary.round(5))
    print({
        'mean_prediction_to_gt_ratio': observation_oof['n_pred'].sum() / observation_oof['n_gt'].sum(),
        'zero_prediction_observations': int((observation_oof['n_pred'] == 0).sum()),
        'mean_fp': observation_oof['fp'].mean(),
        'mean_fn': observation_oof['fn'].mean(),
    })
else:
    observation_oof = annotator_oof = oof_summary = None
    print('OOF evaluation skipped.')

### OOF model-selection rule

Promote a change only if mean observation-level PQ rises, no single fold collapses, and the FP/FN trade-off is understood. Under PQ, a threshold that raises matched-mask Dice while creating many extras can lower the final score.

## Test Inference

### 12. Five-fold instance ensemble

In [ ]:
if KAGGLE_DATA_AVAILABLE and RUN_TEST:
    submission = run_test_ensemble(paths, cfg, WORK_DIR)
    test_ids = {path.stem for path in paths['test'].glob('*.jpeg')}
    submitted_ids = set(submission['filament_id'].str.rsplit('_', n=1).str[0])
    assert submitted_ids == test_ids, f'Missing test image predictions: {sorted(test_ids - submitted_ids)[:5]}'
    assert submission['filament_id'].is_unique
    assert submission['segmentation_rle'].str.len().gt(0).all()
    print({
        'submission_path': str(WORK_DIR / 'submission.csv'),
        'rows': len(submission),
        'covered_images': len(submitted_ids),
        'test_images': len(test_ids),
        'mean_instances_per_image': len(submission) / len(test_ids),
    })
    display(submission.head())
else:
    submission = None
    print('Test inference skipped.')

## Takeaways

- The training and validation unit is the physical JPEG, never the COCO `image_id`.
- PQ, SQ, RQ, FP, and FN—not crop Dice or YOLO mAP—select the final pipeline.
- The seed mask is an explicit refiner input, and training seeds are jittered to reduce the GT-crop/inference-crop gap.
- Five-fold test fusion requires support from at least two folds, except very high-confidence singletons; this is intended to control PQ false positives.
- `submission.csv`, fold manifest, checkpoints, config fingerprint, and OOF tables are written under `/kaggle/working/solar_filament_pq`.

This notebook makes no claim about an executed full OOF score until the full Kaggle run has completed.